# Disaster Damage Assessment
**PyGeoVision v2.0 — Notebook 09**

## Real-World Problem
After the February 2023 Kahramanmaraş earthquake (M7.8), emergency responders
need rapid building damage maps to prioritize search-and-rescue efforts.

```bash
pip install "pygeovision[geo,train]"
```

In [ ]:
import pygeovision as pgv
import numpy as np, rasterio, matplotlib.pyplot as plt
import matplotlib.colors as mcolors, matplotlib.patches as mpatches
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE_DIR = Path("./outputs/09_disaster_damage")
BASE_DIR.mkdir(parents=True, exist_ok=True)

BBOX  = (36.10, 36.50, 37.50, 37.50)   # Kahramanmaraş, Turkey
EVENT = "Turkey-Syria Earthquake, M7.8 — February 6, 2023"
client = pgv.PyGeoVision()
print(f"Event: {EVENT}")

## Step 1: Pre/Post-Event Imagery

In [ ]:
def get_imagery(date_range, label):
    results = client.search(
        bbox=BBOX, date_range=date_range,
        providers=["planetary_computer"], cloud_cover_max=20,
        sort_by="cloud_cover", limit=5,
    )
    print(f"  {label}: {len(results)} scenes (best cloud={results[0].cloud_cover:.0f}%)" if results else f"  {label}: none")
    if results:
        dl = client.download(results[:1], output_dir=BASE_DIR/label.replace(" ","_"),
                              post_process=["reproject:EPSG:32637","cog"])
        if dl and dl[0].success:
            return dl[0].path
    return None

pre_path  = get_imagery(("2022-12-01","2023-02-05"), "pre_event")
post_path = get_imagery(("2023-02-08","2023-02-28"), "post_event")
print(f"\nPre  : {pre_path or 'not available'}")
print(f"Post : {post_path or 'not available'}")

## Step 2: 4-Class Damage Classification

In [ ]:
from pygeovision.models.change_detection.changeformer import ChangeFormer

# xBD-style damage classification:
# Class 0 = No damage     (structure intact)
# Class 1 = Minor damage  (minor cracks, broken windows)
# Class 2 = Major damage  (partial collapse, heavy damage)
# Class 3 = Destroyed     (complete collapse)

DAMAGE_CLASSES = {0:"No damage", 1:"Minor", 2:"Major", 3:"Destroyed"}
DAMAGE_COLORS  = {0:"#2ECC71",   1:"#F39C12", 2:"#E74C3C", 3:"#1C2833"}

print("Building Damage Classification (xBD protocol):")
print()
for cls, name in DAMAGE_CLASSES.items():
    print(f"  Class {cls}: {name:<12} {DAMAGE_COLORS[cls]}")
print()

cd = ChangeFormer(num_classes=4, in_channels=4, device="cpu")

if pre_path and post_path and Path(pre_path).exists() and Path(post_path).exists():
    result     = cd.detect(pre_path, post_path, output_path=str(BASE_DIR/"damage_map.tif"))
    print(f"ChangeFormer result: {result}")
    DAMAGE_SRC = "model"
else:
    print("Generating demo damage map (no scenes available)...")
    H, W = 256, 256
    np.random.seed(42)
    damage_map = np.zeros((H,W), dtype=np.uint8)
    rng = np.random.default_rng(42)
    for _ in range(60):
        r,c  = rng.integers(10,H-35,size=2)
        rh,rw= rng.integers(5,25,size=2)
        lvl  = rng.choice([1,2,3], p=[0.45,0.35,0.20])
        damage_map[r:r+rh, c:c+rw] = lvl
    import rasterio.transform as rt
    cp = BASE_DIR/"damage_map.tif"
    with rasterio.open(cp,'w',driver='GTiff',height=H,width=W,count=1,dtype='uint8',
                       crs='EPSG:32637',transform=rt.from_bounds(*BBOX,W,H)) as dst:
        dst.write(damage_map[np.newaxis])
    DAMAGE_SRC = "demo"
    print(f"Demo map saved: {cp}")

## Step 3: Damage Statistics

In [ ]:
DAMAGE_PATH = BASE_DIR / "damage_map.tif"

if DAMAGE_PATH.exists():
    with rasterio.open(DAMAGE_PATH) as src:
        damage_map = src.read(1)
else:
    np.random.seed(0)
    damage_map = np.random.choice([0,1,2,3],(256,256),p=[0.58,0.22,0.13,0.07])

PIXEL_M2 = 100   # 10m x 10m pixels
total_px = damage_map.size

print("=== RAPID DAMAGE ASSESSMENT REPORT ===")
print(f"Event   : {EVENT}")
print(f"Date    : 2023-02-08 (48h after earthquake)")
print()
print(f"{'Class':<14} {'Area%':>7} {'Area km2':>10} {'Priority':>10}")
print("─" * 48)
damage_pct = {}
for cls, name in DAMAGE_CLASSES.items():
    pct  = float((damage_map == cls).mean() * 100)
    km2  = int((damage_map == cls).sum()) * PIXEL_M2 / 1e6
    prio = {"No damage":"LOW","Minor":"MEDIUM","Major":"HIGH","Destroyed":"CRITICAL"}[name]
    damage_pct[cls] = pct
    print(f"  {name:<12}  {pct:>6.1f}%  {km2:>9.2f}  {prio:>10}")

affected_pct = sum(damage_pct[i] for i in [1,2,3])
critical_pct = damage_pct.get(3, 0)
print(f"\nTotal affected : {affected_pct:.1f}%")
print(f"Critical zones : {critical_pct:.1f}%  <- Priority rescue areas")

## Step 4: Visualisation

In [ ]:
cmap_dmg = mcolors.ListedColormap(list(DAMAGE_COLORS.values()))
norm_dmg = mcolors.BoundaryNorm([0,1,2,3,4], cmap_dmg.N)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im = axes[0].imshow(damage_map, cmap=cmap_dmg, norm=norm_dmg)
axes[0].set_title("Building Damage Map\n4-class severity (ChangeFormer + xBD protocol)",
                   fontsize=11, fontweight='bold')
axes[0].axis('off')
cbar = plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
cbar.set_ticks([0.5,1.5,2.5,3.5])
cbar.set_ticklabels(list(DAMAGE_CLASSES.values()))

# Bar chart
pcts   = [damage_pct.get(i,0) for i in DAMAGE_CLASSES]
bars   = axes[1].bar(list(DAMAGE_CLASSES.values()), pcts,
                      color=list(DAMAGE_COLORS.values()), edgecolor='black', linewidth=0.8)
axes[1].set_ylabel("Area (%)")
axes[1].set_title("Damage Distribution", fontsize=11, fontweight='bold')
axes[1].set_ylim(0, max(pcts)*1.2)
for bar, val in zip(bars, pcts):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                  f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle(f"Emergency Damage Assessment\n{EVENT}", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR/"damage_assessment.png", dpi=150, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 09 — Disaster Damage Assessment")
print("=" * 60)
print(f"Event:     {EVENT}")
print(f"Affected:  {affected_pct:.1f}% of study area")
print(f"Critical:  {critical_pct:.1f}% (destroyed buildings)")
print()
print("Response guidelines (Copernicus EMS):")
print("  Destroyed zones: immediate SAR teams")
print("  Major damage   : structural assessment needed")
print("  Minor damage   : monitoring + repair prioritization")
print()
print("Next: 10_urban_growth_analysis.ipynb")